# Week 5 Lab – Project Ideation

<a href="https://colab.research.google.com/github/bradleyboehmke/uc-bana-4080/blob/main/labs/lab-05-project-ideation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## Overview

This lab is the launchpad for your midterm project. Rather than analyzing data someone else defined, you will identify a business question you care about, figure out how to answer it with the Complete Journey data, and pitch your idea to the class.

By the end of today, each group will have:
- Written out **three distinct project ideas**, each with a clear problem statement, a data plan, and a sense of what a good finding would look like
- Done a **2–3 minute stand-up presentation** to the class on your most compelling idea

There is no analysis required today — the goal is to *think before you code*.

---

### Step 0: Form Your Group and Sign Up on Canvas

Before you open the data, find your project partners. Groups are **2–4 students**.

Once you have agreed on a group go to **Canvas → People → Mid-Term Project** and opt into one of the pre-defined groups (you may not create your own group outside that list). All project submissions — including this week's homework — will be made through your Canvas group.

---

### What Makes a Good Project Question?

A good project question has three qualities:

1. **Specific enough to direct an analysis.** “How do shoppers behave?” is too broad. “Does purchase frequency differ meaningfully across income brackets?” is something you can actually answer with data.

2. **Open enough to allow discovery.** The best EDA questions have a hypothesis but leave room to be surprised. If you already know the answer, there is no analysis to do.

3. **Connected to a real decision.** Someone — a store manager, a marketing director, a merchandiser — should be able to do something different based on your findings. Ask yourself: *if my analysis turns out to be true, what should the retailer do next?*

The Complete Journey dataset includes transactions, products, demographics, coupons, campaigns, and more. You have more data than you need — the constraint is picking the right question, not finding enough data to answer it.

Some starting angles to spark your thinking:
- **Customer behavior:** frequency, loyalty, basket size, seasonal patterns
- **Product performance:** category trends, promotional lift, price sensitivity
- **Demographic segmentation:** do different household types behave differently?
- **Promotion effectiveness:** do coupons change what or how much people buy?
- **Store patterns:** are there differences in how customers shop across stores?

---

## Worked Example: What a Strong Idea Looks Like

Before you start brainstorming, read through this complete example. It shows what each section of the template should look like when done well — and why the specificity matters.

### Example: Do Coupons Actually Drive More Spending — Or Just Lower the Bill?

#### Business Problem

Households that redeem coupons clearly pay less per item — but does the total basket actually get bigger, or are they simply substituting discounted items for things they would have bought anyway? Specifically: do households that regularly redeem coupons spend *more* per shopping trip overall compared to trips where no coupon is used, and does this relationship differ across income brackets?

> **Compare this to a weak version:** *"We want to understand how promotions affect customer behavior."* That version cannot be answered — it has no unit of analysis, no metric, and no comparison to make. The strong version above is specific enough that you would know when you had answered it, and open enough that the data could genuinely surprise you.

#### Decision-Maker Value

The primary audience is the **head of promotions** and the **VP of Marketing**. Their core decision is how to allocate the promotional budget — how many coupons to distribute, to whom, and for which products.

The answer to this question changes that decision substantially:

- If coupons **lift basket size** (people spend more on top of the discounted item), the strategy should maximize redemption rates — distribute more coupons, make them easier to use, and target high-frequency shoppers
- If coupons are **pure substitution** (same basket, just cheaper), the retailer is giving away margin without gaining revenue — the promotional ROI is negative and the budget should be reallocated
- If the effect **differs by income bracket**, targeted promotions by segment become the right play — a single coupon strategy optimized for one group may actively hurt results for another

#### Data Plan

- **Datasets needed:** `transactions`, `coupon_redemptions`, `coupons`, `demographics`
- **Key joins:**
  - Aggregate `transactions` to the basket level (one row per `basket_id`, total spend)
  - Join `coupon_redemptions` on `basket_id` to flag which baskets included a redemption
  - Join `demographics` on `household_id` to bring in income bracket
- **Key derived metrics:**
  - *Coupon trip flag* — did this basket include at least one redemption? (binary, per basket)
  - *Redemption frequency* — how many coupon trips has this household made across all 53 weeks? (used to classify heavy vs. light coupon users at the household level)
  - *Spend lift* — difference in average basket spend between redemption and non-redemption trips, computed within households so each household serves as its own comparison

#### Analysis Approaches

1. **Distribution check** — histogram of basket spend for coupon vs. non-coupon trips side by side; the question is whether the centers and shapes differ
2. **Group comparison** — boxplot of median basket spend: coupon trips vs. non-coupon trips across all households
3. **Household-level segmentation** — classify households into heavy/moderate/light coupon users by redemption frequency; compare their average basket spend to non-users
4. **Income interaction** — repeat the coupon vs. no-coupon comparison within each income bracket; look for whether the spend lift is consistent across income levels or reverses at certain brackets
5. **Category breakdown** — which product departments see the biggest redemptions, and does heavy coupon use in one category crowd out spending in others?

#### What Success Looks Like

- *Expected:* Coupon trips have moderately higher basket spend — the discount incentivizes a slightly larger shop
- *More interesting:* The lift is near zero or negative — coupons are pure substitution, and the promotional budget is eroding margin without driving volume
- *Most valuable finding:* The relationship differs by income. Lower-income households show substitution (same basket, lower cost), while higher-income households show additive spend (coupon trip + more items). This would directly justify an income-targeted promotional strategy and reshape how campaigns are designed and measured.


> **What makes this example work:** every section connects to the others. The question defines the comparison (coupon vs. non-coupon trips). The decision-maker section explains why each possible answer implies a different strategy. The data plan names the exact tables and join keys needed to make that comparison. The analyses build from simple to complex, ending with the income interaction that would make the finding most actionable. The success criteria are specific enough to recognize a meaningful finding — versus just confirming the obvious.

Now use the templates below to develop your own three ideas with the same level of specificity.

## Setup

Run this cell to load the data. Use the exploration cells throughout this notebook to poke around the dataset as you think through your ideas — you don't need to build anything polished today.

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from completejourney_py import get_data

cj_data = get_data()
transactions  = cj_data['transactions']
products      = cj_data['products']
demographics  = cj_data['demographics']
campaigns     = cj_data['campaigns']
coupons       = cj_data['coupons']
coupon_redemptions = cj_data['coupon_redemptions']

# What tables are available?
print("transactions:",  transactions.shape)
print("products:",      products.shape)
print("demographics:",  demographics.shape)
print("campaigns:",     campaigns.shape)
print("coupons:",       coupons.shape)
print("coupon_redemptions:", coupon_redemptions.shape)

In [ ]:
# Explore column names across tables — useful for planning joins
for name, df in [('transactions', transactions), ('products', products),
                  ('demographics', demographics), ('coupons', coupons)]:
    print(f"{name}: {list(df.columns)}")
    print()

---

## Project Idea #1

Fill in each section below. You can edit the markdown cells directly — double-click to open them for editing.

### Idea Title

*Replace this line with a short, descriptive title for your idea (e.g., "Promotional Sensitivity by Income Tier").*

---

**Business Problem**

> *What specific question are you trying to answer? Be precise enough that you'd know when you've answered it. Start with: "We want to understand whether..." or "We want to know how..."*

*Write your problem statement here.*

---

**Decision-Maker Value**

> *Who at the retailer benefits from this analysis? What decision does it enable? What would they do differently if your finding is true?*

*Write here.*

---

**Data Plan**

> *Which datasets do you need? What joins are required? What columns or aggregations are central to your analysis?*

- Datasets needed: 
- Key join(s): 
- Key columns / aggregations: 

---

**Analysis Approaches**

> *What types of analysis would you run? Think about distributions, segmentation, group comparisons, trends over time, correlations.*

*Write here.*

---

**What Success Looks Like**

> *What would a compelling finding reveal? Describe both what you expect to find AND what would genuinely surprise you.*

*Write here.*

In [ ]:
# Quick exploration for Idea #1 — use this to check whether your data plan is feasible
# Your code here

In [ ]:
# Your code here

---

## Project Idea #2

Fill in each section below.

### Idea Title

*Replace this line with a short, descriptive title.*

---

**Business Problem**

*Write your problem statement here.*

---

**Decision-Maker Value**

*Write here.*

---

**Data Plan**

- Datasets needed: 
- Key join(s): 
- Key columns / aggregations: 

---

**Analysis Approaches**

*Write here.*

---

**What Success Looks Like**

*Write here.*

In [ ]:
# Quick exploration for Idea #2
# Your code here

In [ ]:
# Your code here

---

## Project Idea #3

Fill in each section below.

### Idea Title

*Replace this line with a short, descriptive title.*

---

**Business Problem**

*Write your problem statement here.*

---

**Decision-Maker Value**

*Write here.*

---

**Data Plan**

- Datasets needed: 
- Key join(s): 
- Key columns / aggregations: 

---

**Analysis Approaches**

*Write here.*

---

**What Success Looks Like**

*Write here.*

In [ ]:
# Quick exploration for Idea #3
# Your code here

In [ ]:
# Your code here

---

## Presentation Prep

Each group will present their **most compelling idea** to the class in 2–3 minutes. Your presentation should cover:

1. **The question** — what are you trying to understand? (1 sentence)
2. **Why it matters** — who is the decision-maker and what would they do with your findings?
3. **The plan** — what data and analyses would you use?
4. **What you'd hope to find** — what would a meaningful result look like?

You do not need slides. You do not need any analysis. You are pitching an idea.

Use the cell below to jot down your talking points.

### Our Presentation Notes

**Idea we're presenting:** *[Title]*

**The question (1 sentence):**

*Write here.*

**Why it matters:**

*Write here.*

**Our data and analysis plan:**

*Write here.*

**What a good finding would look like:**

*Write here.*

---

## What's Next

After today's lab and presentations, your group will choose **one idea** to develop into a formal project proposal — that is the Week 5 homework assignment. In that proposal, you will go deeper on the problem statement, lay out a specific analysis plan, and begin preliminary exploration of the data.

You are not locked into the idea you present today. If another group's presentation sparks a better direction, you can pivot before submitting the proposal.